<a href="https://colab.research.google.com/github/nilaynishant/acrs/blob/main/00_hello_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Overview

In this notebook we load a **25 km × 25 km Sentinel-2 scene** centred on **Amity University, Noida** for **January 2023** directly from the cloud-hosted AWS data repository via the STAC catalog.

We will:
- Query the Sentinel-2 STAC catalog on AWS to find cloud-free January 2023 scenes
- Load four spectral bands (Blue, Green, Red, NIR) as an XArray Dataset
- Visualise each band individually as a subplot
- Display a **natural-colour RGB** layer-stack composite
- Display a **false-colour NIR** composite to highlight vegetation

### Setup

Determine the runtime environment.

In [ ]:
import os

if 'COLAB_RELEASE_TAG' in os.environ:
    environment = 'colab'
    if os.environ.get('VERTEX_PRODUCT') == 'COLAB_ENTERPRISE':
        environment = 'colab_enterprise'
else:
    environment = 'local'

print(f'Environment: {environment}')

If running on Google Colab, install the required packages.

In [ ]:
%%capture
if environment in ['colab', 'colab_enterprise']:
    !pip install pystac-client odc-stac rioxarray dask['distributed'] botocore

Import all required libraries.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pystac_client
import xarray as xr
import rioxarray as rxr
from odc.stac import configure_s3_access, load

### Study Area: Amity University, Noida

We define a **25 km × 25 km bounding box** centred on Amity University, Noida (Sector 125). The conversion factor of 1° ≈ 111 km is used to convert the radius from kilometres to degrees.

In [ ]:
latitude  = 28.5449   # Amity University, Noida
longitude = 77.3332

# 25 km × 25 km bounding box
km2deg = 1.0 / 111
r = 12.5 * km2deg  # half-width in degrees

bbox = (longitude - r, latitude - r, longitude + r, latitude + r)
print(f'Centre  : {latitude}°N, {longitude}°E')
print(f'BBox    : {bbox}')
print(f'Coverage: {2 * r * 111:.1f} km × {2 * r * 111:.1f} km')

### Search and Load Sentinel-2 Data

We query the [Earth Search STAC catalog](https://earth-search.aws.element84.com/v1) on AWS for **Sentinel-2 Collection 1 Level-2A** scenes in January 2023 that intersect our bounding box and have less than 30% cloud cover.

STAC (Spatio-Temporal Asset Catalog) is an open standard for discovering geospatial datasets. The data is stored on AWS S3 and streamed on demand — no manual download needed.

In [ ]:
catalog = pystac_client.Client.open(
    'https://earth-search.aws.element84.com/v1')

configure_s3_access(aws_unsigned=True)

search = catalog.search(
    collections=['sentinel-2-c1-l2a'],
    bbox=bbox,
    datetime='2023-01',
    query={'eo:cloud_cover': {'lt': 30}},
)
items = search.item_collection()
print(f'Found {len(items)} scene(s) for January 2023')

Load all matching scenes as an XArray Dataset at **30 m resolution**. Using `groupby='solar_day'` ensures images from the same day are merged into a single time step.

In [ ]:
ds = load(
    items,
    bands=['blue', 'green', 'red', 'nir'],
    resolution=30,
    crs='utm',
    bbox=bbox,
    chunks={},
    groupby='solar_day',
)
ds

Compute (download) the data from the cloud repository into memory.

In [ ]:
%%time
ds = ds.compute()

### Explore the Dataset

In [ ]:
print(f'Time steps  : {ds.time.size}')
print(f'Spatial size: {ds.dims["y"]} × {ds.dims["x"]} pixels at 30 m')
print(f'Bands       : {list(ds.data_vars)}')
ds

If more than one scene was found, take a **median composite** across the time dimension to reduce any residual cloud or haze.

In [ ]:
median = ds.median(dim='time')
print('Median composite shape:', dict(median.dims))

### Individual Spectral Bands

Sentinel-2 records reflectance at different wavelengths. Displaying each band separately reveals what information each wavelength captures:

| Band | Wavelength | Highlights |
|------|------------|------------|
| Blue  (B02) | 490 nm | Water bodies, haze |
| Green (B03) | 560 nm | Vegetation peak reflectance |
| Red   (B04) | 665 nm | Chlorophyll absorption |
| NIR   (B08) | 842 nm | Healthy vegetation, biomass |

In [ ]:
band_labels = {
    'blue' : 'Blue  (B02 – 490 nm)',
    'green': 'Green (B03 – 560 nm)',
    'red'  : 'Red   (B04 – 665 nm)',
    'nir'  : 'NIR   (B08 – 842 nm)',
}

fig, axes = plt.subplots(2, 2)
fig.set_size_inches(12, 10)

for ax, (band, label) in zip(axes.flat, band_labels.items()):
    median[band].plot.imshow(ax=ax, cmap='gray', robust=True)
    ax.set_title(label, fontsize=11)
    ax.set_axis_off()
    ax.set_aspect('equal')

plt.suptitle('Sentinel-2 Spectral Bands – Amity University, Noida (Jan 2023)', fontsize=13)
plt.tight_layout()
plt.show()

### Natural Colour (RGB) Layer Stack

Stacking the **Red, Green, and Blue** bands into the three colour channels produces a **natural-colour composite** — this is roughly what the human eye would see from space. Roads, buildings and bare soil appear grey-brown; dense vegetation appears green.

In [ ]:
median_da = median.to_array('band')

fig, ax = plt.subplots(1, 1)
fig.set_size_inches(8, 8)
median_da.sel(band=['red', 'green', 'blue']).plot.imshow(
    ax=ax, robust=True)
ax.set_title('Natural Colour RGB – Amity University, Noida (Jan 2023)')
ax.set_axis_off()
ax.set_aspect('equal')
plt.show()

### False Colour Composite (NIR – Red – Green)

In a **false-colour composite**, the NIR band is mapped to the red display channel. Healthy vegetation reflects very strongly in NIR and appears **bright red**, making it easy to distinguish:
- **Red / magenta** → healthy vegetation
- **Grey / white** → urban surfaces, bare soil
- **Dark blue / black** → water

In [ ]:
fig, ax = plt.subplots(1, 1)
fig.set_size_inches(8, 8)
median_da.sel(band=['nir', 'red', 'green']).plot.imshow(
    ax=ax, robust=True)
ax.set_title('False Colour Composite (NIR-R-G) – Amity University, Noida (Jan 2023)')
ax.set_axis_off()
ax.set_aspect('equal')
plt.show()